# Solutions · 4. Smooth monotone fits with I-splines

Worked solutions to the three exercises of [Lecture 4](../../notebooks/part2_shape/04_monotone_splines_and_convex_free_energy_potentials.ipynb). Each solution is computed here and checked with `assert` statements against an exact identity or an independent calculation. The first code cell sets up the notebook and repeats the demonstrations of the lecture without printing them, because the solutions use their variables.

In [1]:
try:                      # use the installed package, or the copy in this repository
    import engml_2_applied
except ImportError:
    import pathlib, subprocess, sys
    for _root in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_root / "engml_2_applied" / "__init__.py").exists():
            sys.path.insert(0, str(_root))
            break
    else:                 # on Google Colab: install from GitHub
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "git+https://github.com/ktwyw/engml-2-applied.git"], check=True)
    import engml_2_applied

import inspect
import numpy as np
import matplotlib.pyplot as plt
from engml_2_applied import (clustering_phases, constants, descriptors, dim_reduction, linear_physics,
                             trees_ensembles, uncertainty)
from engml_2_applied.plotting import panel_label, use_style

use_style()

def show_source(obj):
    """Return the source code of a library function or class."""
    return inspect.getsource(obj)

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

import io, sys
_stdout, sys.stdout = sys.stdout, io.StringIO()   # the demonstrations are repeated without their output
heading("Demonstration 1: Non-Negative M-Spline & Cumulative I-Spline Basis Verification")
x_grid = np.linspace(0.0, 10.0, 101)
knots = np.linspace(0.0, 10.0, 8)
I_mat, M_mat = linear_physics.ispline_basis_1d(x_grid, knots)

eps = 1e-5
I_plus, _ = linear_physics.ispline_basis_1d(x_grid + eps, knots)
I_minus, _ = linear_physics.ispline_basis_1d(x_grid - eps, knots)
dI_fd = (I_plus - I_minus) / (2.0 * eps)

print(f"Basis matrix shapes: I_mat = {I_mat.shape}, M_mat = {M_mat.shape}")
print(f"M-spline range     : [{np.min(M_mat):.4e}, {np.max(M_mat):.4f}] (Strictly non-negative)")
print(f"I-spline range     : [{np.min(I_mat):.4f}, {np.max(I_mat):.4f}] (Strictly bounded in [0, 1])")
print(f"Max derivative check ||dI/dx - M||_max: {np.max(np.abs(dI_fd - M_mat)):.2e}")
assert np.all(M_mat >= 0.0) and np.all((I_mat >= 0.0) & (I_mat <= 1.0))
assert np.allclose(dI_fd, M_mat, atol=1e-7)
heading("Demonstration 2: Projected Coordinate Descent & Non-Negative Coefficient Spectrum")
rng = np.random.default_rng(42)
P_bar = np.linspace(0.1, 10.0, 50)
q_true = 5.2 * 1.4 * P_bar / (1.0 + 1.4 * P_bar)
q_obs = q_true + rng.normal(0.0, 0.18, size=50)

knots_ads = np.linspace(0.1, 10.0, 9)
isp_fit = linear_physics.fit_monotone_ispline(P_bar, q_obs, knots=knots_ads, increasing=True)
print("Intercept beta_0 :", round(float(isp_fit["beta"][0]), 4))
print("Non-negative I-spline weights beta_{1..9}:", np.round(isp_fit["beta"][1:], 4))
print(f"Min analytical slope dq/dP across [0.1, 10] bar: {np.min(isp_fit['dy_dx']):.4e} >= 0")
assert np.all(isp_fit["beta"][1:] >= 0.0) and np.all(isp_fit["dy_dx"] >= -1e-15)
heading("Demonstration 3: Value & Derivative Benchmark — I-Spline vs. PAVA vs. Polynomial")
dq_dP_true = 5.2 * 1.4 / ((1.0 + 1.4 * P_bar) ** 2)
q_pava_ads = linear_physics.pava_monotone_regression(q_obs, increasing=True)
dq_dP_pava = np.gradient(q_pava_ads, P_bar)

poly8 = np.polyfit(P_bar, q_obs, deg=8)
q_poly8 = np.polyval(poly8, P_bar)
dq_dP_poly8 = np.polyval(np.polyder(poly8), P_bar)

for name, q_m, dq_m in [("Unconstrained Poly-8", q_poly8, dq_dP_poly8), ("Staircase PAVA", q_pava_ads, dq_dP_pava), ("Smooth I-Spline", isp_fit["y_pred"], isp_fit["dy_dx"])]:
    v_rmse = float(np.sqrt(np.mean((q_m - q_true) ** 2)))
    d_rmse = float(np.sqrt(np.mean((dq_m - dq_dP_true) ** 2)))
    n_viol = int(np.sum(dq_m < -1e-12))
    print(f"{name:22s} | Value RMSE = {v_rmse:.4f} | Slope RMSE = {d_rmse:.4f} | Slope Violations = {n_viol}")
assert np.sqrt(np.mean((isp_fit["y_pred"] - q_true) ** 2)) < 0.12
assert int(np.sum(isp_fit["dy_dx"] < -1e-12)) == 0 and int(np.sum(dq_dP_poly8 < -1e-12)) > 0
heading("Demonstration 4: Knot Density Sweep & Convex Helmholtz Free-Energy Curvature")
for n_k in [4, 8, 14, 22]:
    fit_k = linear_physics.fit_monotone_ispline(P_bar, q_obs, knots=np.linspace(0.1, 10.0, n_k), increasing=True)
    v_err = float(np.sqrt(np.mean((fit_k["y_pred"] - q_true) ** 2)))
    print(f"Knots M = {n_k:2d} | Active positive weights = {int(np.sum(fit_k['beta'][1:] > 1e-5)):2d} | Value RMSE = {v_err:.4f} | Min slope = {np.min(fit_k['dy_dx']):.2e}")

Vm = np.linspace(0.15, 0.80, 45)
P_iso_noisy = 25.0 / Vm + rng.normal(0.0, 1.5, size=45)
isp_p = linear_physics.fit_monotone_ispline(Vm, P_iso_noisy, knots=np.linspace(0.15, 0.80, 8), increasing=False)
d2A_dVm2 = -isp_p["dy_dx"]
print(f"\nMin Helmholtz free-energy curvature d^2 A_m / dV_m^2 = {np.min(d2A_dVm2):.4e} >= 0 (Strictly convex)")
assert np.all(d2A_dVm2 >= -1e-15)
heading("Inside the Algorithm: By-Hand Projected Coordinate Descent vs. the Library")
I_b, M_b = linear_physics.ispline_basis_1d(P_bar, knots_ads)
Phi = np.column_stack([np.ones_like(P_bar), I_b])
n_s, p_s = Phi.shape
beta_hand = np.zeros(p_s)
beta_hand[0] = float(np.mean(q_obs))
col_sq = np.sum(Phi ** 2, axis=0) + np.concatenate([[0.0], np.full(p_s - 1, 1e-6 * n_s)])
res_vec = q_obs - Phi @ beta_hand
for _ in range(800):
    b_prev = beta_hand.copy()
    for j in range(p_s):
        old_v = beta_hand[j]
        rho = float(np.dot(Phi[:, j], res_vec)) + float(np.sum(Phi[:, j] ** 2)) * old_v
        new_v = (rho / col_sq[j]) if j == 0 else max(0.0, rho / col_sq[j])
        if new_v != old_v:
            res_vec -= Phi[:, j] * (new_v - old_v)
            beta_hand[j] = new_v
    if np.max(np.abs(beta_hand - b_prev)) < 1e-10:
        break
diff_isp = float(np.max(np.abs(beta_hand - isp_fit["beta"])))
print("by hand (pure NumPy) beta[:5]:", np.round(beta_hand[:5], 6))
print("library (engml_2_applied)      beta[:5]:", np.round(isp_fit["beta"][:5], 6))
print(f"Max digit-for-digit discrepancy: {diff_isp:.2e}")
assert diff_isp < 1e-13
sys.stdout = _stdout
print("Set-up complete: the variables of the lecture's demonstrations are defined.")

Set-up complete: the variables of the lecture's demonstrations are defined.


## Exercise 1

**Problem.** Fit a decreasing smooth monotone I-spline (`increasing=False`) to noisy shear-thinning polymer viscosity measurements $\log_{10}\eta(\dot{\gamma})$ across $\log_{10}\dot{\gamma} \in [-1, 3]$ generated from a Carreau–Yasuda fluid $\eta(\dot{\gamma}) = \eta_0 (1 + (\lambda\dot{\gamma})^2)^{(n-1)/2}$ ($\eta_0 = 1000\;\mathrm{Pa\cdot s}, \lambda = 1.0\;\mathrm{s}, n = 0.35$) with noise $\sigma = 0.08$. Verify that the fitted slope $d(\log_{10}\hat{\eta})/d(\log_{10}\dot{\gamma}) \le 0$ at every point and that the test RMSE against the true viscosity curve is below $0.05$.

In [2]:
heading("Exercise 1: Carreau-Yasuda Shear-Thinning Polymer Viscosity I-Spline")
rng_rheo = np.random.default_rng(19)
log_gdot = np.linspace(-1.0, 3.0, 45)
gdot = 10.0 ** log_gdot
log_eta_true = np.log10(1000.0 * ((1.0 + gdot ** 2) ** ((0.35 - 1.0) / 2.0)))
log_eta_noisy = log_eta_true + rng_rheo.normal(0.0, 0.08, size=45)

rheo_isp = linear_physics.fit_monotone_ispline(log_gdot, log_eta_noisy, knots=np.linspace(-1.0, 3.0, 8), increasing=False)
rmse_rheo = float(np.sqrt(np.mean((rheo_isp["y_pred"] - log_eta_true) ** 2)))
print(f"Shear-thinning I-spline RMSE = {rmse_rheo:.4f} | Max slope = {np.max(rheo_isp['dy_dx']):.4e} <= 0")
assert np.all(rheo_isp["dy_dx"] <= 1e-15) and rmse_rheo < 0.05


Exercise 1: Carreau-Yasuda Shear-Thinning Polymer Viscosity I-Spline
Shear-thinning I-spline RMSE = 0.0300 | Max slope = -2.7180e-02 <= 0


**Discussion.** Non-negative I-spline weights with `increasing=False` guarantee $d(\log_{10}\eta)/d(\log_{10}\dot{\gamma}) \le 0$ across both the Newtonian plateau and the power-law shear-thinning regime.

---

## Exercise 2

**Problem.** Compare Newton–Raphson inversion $\hat{q}(P^*) = q_{\mathrm{target}} = 3.0\;\mathrm{mol/kg}$ on the smooth I-spline surrogate (`isp_fit`) versus the staircase PAVA curve. Show that Newton–Raphson on the smooth I-spline converges to $|\hat{q}(P^*) - 3.0| < 10^{-12}$ in $\le 6$ iterations.

In [3]:
heading("Exercise 2: Newton-Raphson Process Inversion on Smooth I-Spline Surrogate")
q_target = 3.0
P_curr = 2.0
b_isp = isp_fit["beta"]
for it in range(1, 8):
    I_c, M_c = linear_physics.ispline_basis_1d(np.array([P_curr]), knots_ads)
    q_val = float(b_isp[0] + I_c[0] @ b_isp[1:])
    dq_val = float(M_c[0] @ b_isp[1:])
    res = q_val - q_target
    P_curr -= res / dq_val
    print(f"Newton iter {it} | P = {P_curr:.8f} bar | q(P) - q_target = {res:+.2e} | slope dq/dP = {dq_val:.4f}")
    if abs(res) < 1e-12:
        break
assert abs(res) < 1e-12 and it <= 6


Exercise 2: Newton-Raphson Process Inversion on Smooth I-Spline Surrogate
Newton iter 1 | P = 0.49271485 bar | q(P) - q_target = +8.99e-01 | slope dq/dP = 0.5966
Newton iter 2 | P = 1.03618246 bar | q(P) - q_target = -1.09e+00 | slope dq/dP = 1.9979
Newton iter 3 | P = 1.10564489 bar | q(P) - q_target = -1.08e-01 | slope dq/dP = 1.5492
Newton iter 4 | P = 1.10738902 bar | q(P) - q_target = -2.57e-03 | slope dq/dP = 1.4748
Newton iter 5 | P = 1.10739013 bar | q(P) - q_target = -1.64e-06 | slope dq/dP = 1.4729
Newton iter 6 | P = 1.10739013 bar | q(P) - q_target = -6.74e-13 | slope dq/dP = 1.4729


**Discussion.** Because the I-spline derivative $dq/dP = \sum_m \beta_m M_m(P) > 0$ is smooth and strictly positive, Newton's method converges in six iterations, with the residual falling from $10^{-3}$ to $10^{-6}$ to $10^{-12}$ in the last three.

---

## Exercise 3

**Problem.** Implement a **C-spline** (convex spline) basis generator `convex_cspline_basis_1d(x, knots)` by integrating the monotone I-spline basis $C_m(x) = \int_{x_0}^x I_m(u)\,du$ via cumulative trapezoidal quadrature so that $C_m''(x) = M_m(x) \ge 0$, and verify that any non-negative combination $\hat{A}(x) = \sum_{m=1}^M \beta_m C_m(x)$ ($\beta_m > 0$) has strictly positive second derivative $\hat{A}''(x) > 0$ everywhere.

In [4]:
heading("Exercise 3: Implement Convex C-Spline Free-Energy Basis From Scratch")
def convex_cspline_basis_1d(x: np.ndarray, knots: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Return convex C-spline basis C_mat, its 1st derivative I_mat, and 2nd derivative M_mat >= 0."""
    xv = np.asarray(x, dtype=float).ravel()
    I_m, M_m = linear_physics.ispline_basis_1d(xv, knots)
    dx = np.diff(xv)[:, None]
    trapz_inc = 0.5 * (I_m[:-1, :] + I_m[1:, :]) * dx
    C_m = np.vstack([np.zeros((1, len(knots))), np.cumsum(trapz_inc, axis=0)])
    return C_m, I_m, M_m

C_b, I_b_c, M_b_c = convex_cspline_basis_1d(Vm, np.linspace(0.15, 0.80, 8))
weights_pos = np.ones(8)
A_potential = C_b @ weights_pos
d2A_num = np.gradient(np.gradient(A_potential, Vm), Vm)
print(f"Min analytical curvature sum(beta_m M_m): {np.min(M_b_c @ weights_pos):.4e} > 0")
print(f"Min numerical 2nd derivative d^2A/dVm^2 : {np.min(d2A_num):.4e} > 0")
assert np.all((M_b_c @ weights_pos) > 0.0) and np.all(d2A_num > 0.0)


Exercise 3: Implement Convex C-Spline Free-Energy Basis From Scratch
Min analytical curvature sum(beta_m M_m): 8.2490e+00 > 0
Min numerical 2nd derivative d^2A/dVm^2 : 4.4597e+00 > 0


**Discussion.** Integrating M-splines once yields monotone I-splines ($I_m' = M_m \ge 0$); integrating twice yields convex C-splines ($C_m'' = M_m \ge 0$) suitable for parameterizing convex thermodynamic free energies $A_m(V_m)$ and $G_m(x)$.